In [ ]:
import json
from pathlib import Path

# Path to your JSON file
json_path = Path("data/thesis_data_1778418853489.json")

# Load the JSON
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("JSON loaded successfully!")
print(type(data))

In [ ]:
import pprint

pp = pprint.PrettyPrinter(depth=3)

pp.pprint(data[0])  # or data if dict

Ok so that worked. The problem now is, as we can see from the pretty print, that each sample has some nested stepData inside. This means we can't just put it in a dataframe and need to process it first.

In [ ]:
import pandas as pd

# Create a simplified version of the data
simplified_data = []

for item in data:
    simplified_item = item.copy()

    # Create a new column with the number of steps
    simplified_item["numSteps"] = len(item.get("stepData", []))

    # Remove the original huge stepData field
    del simplified_item["stepData"]

    simplified_data.append(simplified_item)

# Convert to DataFrame
df = pd.DataFrame(simplified_data)

# Preview
print(df.head())

Ok things are at least graspable now. Ofc this isn't helpful for the project, so lets see what we wanna do with the data.

Next we should crop parts of the steps that have constant size, because neural networks work better with that (or maybe can only work with that even idk)

In [ ]:
import random

WINDOW_SIZE = 60

samples = []

for item in data:

    step_data = item.get("stepData", [])

    # Skip sequences that are too short
    if len(step_data) < WINDOW_SIZE:
        continue

    # Pick random starting point
    max_start = len(step_data) - WINDOW_SIZE
    start_index = random.randint(0, max_start)

    # Extract fixed-size window
    window = step_data[start_index:start_index + WINDOW_SIZE]

    samples.append({
        "fastBuilderMode": item["fastBuilderMode"],
        "idAnonymous": item["idAnonymous"],
        "window": window
    })

print("Number of samples:", len(samples))

# TESTING ONLY
df2 = pd.DataFrame(samples)
df2["window"] = df2["window"].apply(len)
print(df2.head())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Only look at NORMAL mode replays for now
normal_samples = [s for s in samples if s["fastBuilderMode"] == "NORMAL"]

# Positions as an array of shape (samples, WINDOW_SIZE, 3) with x, y, z
positions = np.array([
    [[step["playerPosition"]["x"], step["playerPosition"]["y"], step["playerPosition"]["z"]]
     for step in s["window"]]
    for s in normal_samples
])

# Movement per step: sqrt(dx² + dy² + dz²) relative to the previous step
deltas = np.diff(positions, axis=1)
movement = np.sqrt((deltas ** 2).sum(axis=2))  # shape (samples, WINDOW_SIZE - 1)

print("NORMAL samples:", len(normal_samples))

steps = np.arange(1, WINDOW_SIZE)

plt.figure(figsize=(12, 5))
for m in movement:
    plt.plot(steps, m, color="gray", alpha=0.1, linewidth=0.8)
plt.plot(steps, movement.mean(axis=0), color="tab:blue", linewidth=2, label="mean")
plt.xlabel("step in window")
plt.ylabel("movement (blocks per step)")
plt.title(f"Movement per step, NORMAL mode ({len(normal_samples)} windows)")
plt.legend()
plt.show()

Now lets turn all windows into one "wide" DataFrame that a simple (non-sequential) neural network can take: one row per window, one column per feature per step.

- `movement_1 .. movement_59`: blocks moved since the previous step (like in the plot above)
- `yawChange_*` / `pitchChange_*`: how much the player turned their head since the previous step
- `clicks_0 .. clicks_59`, `sneaking_0 ..`, `blocksChanged_0 ..`: the raw per-step values

The change columns start at 1 because 60 steps only give 59 differences. Absolute positions are left out on purpose, because they mostly say where on the map someone is, not how they play.

In [ ]:
def per_step(get_value):
    """Array of shape (samples, WINDOW_SIZE): get_value(step) for every step of every window."""
    return np.array([[get_value(step) for step in s["window"]] for s in samples], dtype=float)

def step_columns(name, values, first_step):
    """Turn a (samples, steps) array into columns name_<first_step>, name_<first_step + 1>, ..."""
    columns = [f"{name}_{i}" for i in range(first_step, first_step + values.shape[1])]
    return pd.DataFrame(values, columns=columns)

x = per_step(lambda step: step["playerPosition"]["x"])
y = per_step(lambda step: step["playerPosition"]["y"])
z = per_step(lambda step: step["playerPosition"]["z"])
yaw = per_step(lambda step: step["playerPosition"]["yaw"])
pitch = per_step(lambda step: step["playerPosition"]["pitch"])

# Changes between consecutive steps, shape (samples, WINDOW_SIZE - 1)
movement = np.sqrt(np.diff(x, axis=1) ** 2 + np.diff(y, axis=1) ** 2 + np.diff(z, axis=1) ** 2)
# yaw is not limited to -180..180 in the data, so wrap the change to the shortest turn (-180..180)
yaw_change = (np.diff(yaw, axis=1) + 180) % 360 - 180
pitch_change = np.diff(pitch, axis=1)

clicks = per_step(lambda step: step["clicks"])
sneaking = per_step(lambda step: step["sneaking"])  # True/False -> 1.0/0.0
blocks_changed = per_step(lambda step: len(step["blocksChanged"]))

ping = per_step(lambda step: step["ping"])
ping_average = ping.mean(axis=1)
ping_jitter = np.abs(np.diff(ping, axis=1)).mean(axis=1)

wide_df = pd.concat([
    pd.DataFrame({
        "idAnonymous": [s["idAnonymous"] for s in samples], # remove that for clustering
        "fastBuilderMode": [s["fastBuilderMode"] for s in samples], # remove that for clustering
        "pingAvg": ping_average, # remove that for clustering
        "pingJitter": ping_jitter, # remove that for clustering
    }),
    step_columns("movement", movement, 1),
    step_columns("yawChange", yaw_change, 1),
    step_columns("pitchChange", pitch_change, 1),
    step_columns("clicks", clicks, 0),
    step_columns("sneaking", sneaking, 0),
    step_columns("blocksChanged", blocks_changed, 0),
], axis=1)

print(wide_df.shape)  # (samples, 4 + 3 * (WINDOW_SIZE - 1) + 3 * WINDOW_SIZE)

# Preview only: show the id/label columns and steps 0-2 of each feature (wide_df itself keeps all columns)
wide_df.filter(regex=r"^(idAnonymous|fastBuilderMode|pingAvg|pingJitter)$|_[0-2]$").head()

This should do it for now with the data. It's flattened. Technicalities should hereby be done to now look after the conceptual steps of preprocessing

Let's create some plots next to get to know our data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# movement plots

steps = np.arange(1, WINDOW_SIZE)

modes_list = wide_df["fastBuilderMode"].unique() #get a list of all modes
movement_columns = [f"movement_{i}" for i in steps] #create a list [movement_1, movement_2, ...]

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 7), sharex=True, sharey=True)

for mode, ax in zip(modes_list, axes.flat):
    ax.set_ylim(0, 1)

    #get only the rows of {mode} and only the columns from {movement_columns}
    mode_movement = wide_df.loc[wide_df["fastBuilderMode"] == mode, movement_columns].to_numpy()

    amount_of_samples = 20
    for m in mode_movement[:amount_of_samples]:
        ax.plot(steps, m, color="gray", alpha=0.2, linewidth=0.8)

    ax.plot(steps, mode_movement.mean(axis=0), color="tab:red", linewidth=2, label="mean")
    ax.set_title(f"{mode} ({amount_of_samples} / {len(mode_movement)} windows)")

axes.flat[-1].set_visible(False)
axes.flat[0].legend()
fig.supxlabel("tick in window")
fig.supylabel("movement (meters per tick)")
plt.tight_layout()

plt.show()



So we've plotted movement data for each mode separately. Looks cool to have the data, but it's not too helpful yet this way. We can already see some outliers, tho.

In [ ]:
sample = [{**item, "stepData": item["stepData"][:5]} for item in data[:3]]
with open("data/sample_pretty.json", "w", encoding="utf-8") as f:
    json.dump(sample, f, indent=2)

**^ readable sample generator**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# movement average and consistency plot

fig, ax = plt.subplots(figsize=(9,7))

modes_list = wide_df["fastBuilderMode"].unique()
movement_array = wide_df.loc[:, wide_df.columns.str.startswith("movement_")].to_numpy()

movement_avg = movement_array.mean(axis=1)
movement_std = movement_array.std(axis=1)

for mode in modes_list:
    is_mode = (wide_df["fastBuilderMode"] == mode).to_numpy()
    mode_avg = movement_avg[is_mode]
    mode_std = movement_std[is_mode]

    ax.scatter(mode_avg, mode_std, s=1.5, alpha=0.8, label=f"{mode}")

ax.legend(markerscale=5)
ax.set_title("movement & modes")
ax.set_xlabel("movement average [meters per tick]")
ax.set_ylabel("movement standard deviation")
plt.show()


The line graphs didn't seem very clusterable, so I thought about what kind of graph seems better suited for that. It seems sources that are more knowledgable than me suggest to reduce higher-dimensional data to less dimensions, so I thought of doing something similar by deriving few variables from the huge amount of values we have in the data. My first idea was to summorize the movement data with just the average and standard deviation. This lets us have a single data point per replay, which seems much more clusterable to me. I'm not sure yet if this is too much of an information loss for the data to be used meaningfully, but we have to start somewhere, right? I also made it so the data point has a different color depending on which fastbuilder mode it was played it, so we can maybe already see some patterns on the plot (yes we can  indeed see some patterns).

So next we wanna do the same with data other than the movement.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# movement average and consistency plot

fig, ax = plt.subplots(figsize=(9,7))

modes_list = wide_df["fastBuilderMode"].unique()
clicks_array = wide_df.loc[:, wide_df.columns.str.startswith("clicks_")].to_numpy()
blocks_array = wide_df.loc[:, wide_df.columns.str.startswith("blocksChanged_")].to_numpy()

clicks_avg = clicks_array.mean(axis=1)
blocks_avg = blocks_array.mean(axis=1)

for mode in modes_list:
    is_mode = (wide_df["fastBuilderMode"] == mode).to_numpy()
    mode_avg = clicks_avg[is_mode] *5
    mode_std = blocks_avg[is_mode] *5

    ax.scatter(mode_avg, mode_std, s=1.5, alpha=0.8, label=f"{mode}")

ax.legend(markerscale=5)
ax.set_title("clicks/blocks & modes")
ax.set_xlabel("clicks average [clicks per second]")
ax.set_ylabel("blocks placed average [blocks per second]")

plt.xlim(0, 20)
plt.ylim(0, 5)

plt.show()


Experimenting with clicks and blocks placed data, but so far I don't think it's helpful at all sadly

Let's try to approach this differently. Since this is for clustering and not for neural network input, we'll use full replay data again (instead of windows) and calculate the blocks/clicks ratio. My expectation is that there will be a few clusters. Close to 1 block/click might be cheaters who don't click themselves and very slow players. And depending on clicking and bridging techniques, I think more clusters might distinguish themselves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# blocks per click & movement speed, one point per replay
# (each replay is clipped to its longest window without standing still for more than MAX_STILL_TICKS)

STILL_THRESHOLD = 0.01  # moving less than this (meters per tick) counts as standing still
MAX_STILL_TICKS = 3     # standing still for longer than this counts as not playing
MIN_BLOCKS = 20         # replays with fewer blocks give a too coarse ratio

rows = []

for item in data:
    steps = item["stepData"]
    if len(steps) < 2:
        continue

    positions = np.array([[step["playerPosition"]["x"], step["playerPosition"]["y"], step["playerPosition"]["z"]]
                          for step in steps])

    # movement[i] is how far the player moved from step i to step i + 1,
    # so from here on we look at steps[1:], which lines up with movement
    movement = np.sqrt((np.diff(positions, axis=0) ** 2).sum(axis=1))
    is_still = movement < STILL_THRESHOLD

    # keep[i] = False for every tick that is part of a still stretch longer than MAX_STILL_TICKS,
    # these stretches are where we split the replay into windows
    keep = np.ones(len(movement), dtype=bool)
    still_ticks_in_a_row = 0
    for i in range(len(movement)):
        if is_still[i]:
            still_ticks_in_a_row += 1
        else:
            still_ticks_in_a_row = 0

        if still_ticks_in_a_row > MAX_STILL_TICKS:
            # mark the whole stretch so far, including its first ticks
            keep[i - still_ticks_in_a_row + 1 : i + 1] = False

    # find the longest window of kept ticks in a row, so the replay is clipped to one single window
    # (anything outside it, like building a house somewhere, is cut completely)
    best_start, best_length = 0, 0
    window_start, window_length = 0, 0
    for i in range(len(keep)):
        if keep[i]:
            if window_length == 0:
                window_start = i  # a new window starts here
            window_length += 1
            if window_length > best_length:
                best_start, best_length = window_start, window_length
        else:
            window_length = 0

    if best_length == 0:
        continue  # the player never moved, so there is no usable window

    window = slice(best_start, best_start + best_length)

    # only count clicks, blocks and movement inside the window
    clicks = np.array([step["clicks"] for step in steps[1:]])[window].sum()
    blocks = np.array([len(step["blocksChanged"]) for step in steps[1:]])[window].sum()
    window_movement = movement[window]

    if blocks < MIN_BLOCKS or clicks == 0:
        continue

    rows.append({
        "fastBuilderMode": item["fastBuilderMode"],
        "idAnonymous": item["idAnonymous"],
        "blocks_per_click": blocks / clicks,
        "speed": window_movement.mean() * 20,         # meters per tick -> meters per second (20 ticks per second)
        "window_share": best_length / len(movement),  # how much of the replay the window covers
    })

ratio_df = pd.DataFrame(rows)
print("replays used:", len(ratio_df), "/", len(data))
print("average share of a replay covered by its window:", round(ratio_df["window_share"].mean(), 3))

modes_list = ratio_df["fastBuilderMode"].unique()

# plot 1: all modes in one scatter plot

fig, ax = plt.subplots(figsize=(9, 7))

for mode in modes_list:
    mode_rows = ratio_df[ratio_df["fastBuilderMode"] == mode]
    ax.scatter(mode_rows["blocks_per_click"], mode_rows["speed"], s=1.5, alpha=0.8, label=mode)

# Minecraft's movement speeds on flat ground, as a reference
for name, speed in [("sneaking", 1.3), ("walking", 4.3), ("sprinting", 5.6)]:
    ax.axhline(speed, color="gray", linestyle="--", linewidth=0.8)
    ax.text(0.01, speed + 0.05, name, color="gray")

ax.legend(markerscale=5)
ax.set_title("blocks per click & movement speed")
ax.set_xlabel("blocks placed per click")
ax.set_ylabel("average movement speed [meters per second]")
plt.xlim(0, 1.05)
plt.ylim(0, 9)
plt.show()

# plot 2: the same points, one plot per mode (all other modes in light gray behind it to compare)

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 7), sharex=True, sharey=True)

for index, (mode, ax) in enumerate(zip(modes_list, axes.flat)):
    is_mode = ratio_df["fastBuilderMode"] == mode

    ax.scatter(ratio_df.loc[~is_mode, "blocks_per_click"], ratio_df.loc[~is_mode, "speed"],
               s=1, color="lightgray")
    # f"C{index}" is the same color this mode had in plot 1
    ax.scatter(ratio_df.loc[is_mode, "blocks_per_click"], ratio_df.loc[is_mode, "speed"],
               s=1.5, alpha=0.8, color=f"C{index}")
    ax.set_title(f"{mode} ({is_mode.sum()} replays)")

axes.flat[-1].set_visible(False)
fig.supxlabel("blocks placed per click")
fig.supylabel("average movement speed [meters per second]")
plt.xlim(0, 1.05)
plt.ylim(0, 9)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_samples

# DBSCAN clustering on blocks per click & movement speed, separately for every mode

EPS = 0.3                # how close (after scaling) points have to be to count as neighbours
MIN_SAMPLES_SHARE = 0.03 # a cluster needs at least this share of the mode's replays as neighbours

ratio_df["cluster"] = -1  # -1 is what DBSCAN calls noise: points that don't fit any cluster
# how well a point fits its cluster (silhouette score, -1..1):
# close to 1 = clearly in its own cluster, around 0 = between two clusters, below 0 = closer to another cluster
# noise points have no cluster, so they stay empty (NaN)
ratio_df["cluster_fit"] = np.nan

for mode in modes_list:
    is_mode = ratio_df["fastBuilderMode"] == mode
    features = ratio_df.loc[is_mode, ["blocks_per_click", "speed"]]

    # scale both features to the same range, otherwise speed (0..9) would matter much more than the ratio (0..1)
    scaled = StandardScaler().fit_transform(features)

    # modes have very different numbers of replays, so min_samples grows with the mode size (but at least 5)
    min_samples = max(5, int(MIN_SAMPLES_SHARE * is_mode.sum()))
    labels = DBSCAN(eps=EPS, min_samples=min_samples).fit_predict(scaled)
    ratio_df.loc[is_mode, "cluster"] = labels

    # the silhouette score compares clusters with each other, so it needs at least 2 clusters in this mode
    in_cluster = labels != -1
    if len(set(labels[in_cluster])) >= 2:
        mode_index = ratio_df.index[is_mode]  # row labels of this mode in ratio_df
        ratio_df.loc[mode_index[in_cluster], "cluster_fit"] = silhouette_samples(scaled[in_cluster], labels[in_cluster])

# overview: how many replays each cluster has, where its middle (median) is and how well its points fit on average
print(ratio_df.groupby(["fastBuilderMode", "cluster"])
              .agg(replays=("speed", "size"),
                   blocks_per_click=("blocks_per_click", "median"),
                   speed=("speed", "median"),
                   cluster_fit=("cluster_fit", "mean"))
              .round(2))

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 7), sharex=True, sharey=True)

for mode, ax in zip(modes_list, axes.flat):
    mode_rows = ratio_df[ratio_df["fastBuilderMode"] == mode]

    noise = mode_rows[mode_rows["cluster"] == -1]
    ax.scatter(noise["blocks_per_click"], noise["speed"], s=1.5, color="lightgray", label="noise")

    for cluster in range(mode_rows["cluster"].max() + 1):
        cluster_rows = mode_rows[mode_rows["cluster"] == cluster]
        ax.scatter(cluster_rows["blocks_per_click"], cluster_rows["speed"], s=1.5, alpha=0.8,
                   color=f"C{cluster}", label=f"cluster {cluster}")

    ax.set_title(f"{mode} ({mode_rows['cluster'].max() + 1} clusters)")
    ax.legend(markerscale=5, fontsize=7)

axes.flat[-1].set_visible(False)
fig.supxlabel("blocks placed per click")
fig.supylabel("average movement speed [meters per second]")
plt.xlim(0, 1.05)
plt.ylim(0, 9)
plt.tight_layout()
plt.show()

In [ ]:
import json
from pathlib import Path

# export ratio_df to a readable JSON file, grouped like this:
# { mode: { "cluster 0": [replays, best fitting first], "cluster 1": [...], ..., "noise": [...] } }
# data/ is gitignored, so the export stays local like the rest of the data

export_path = Path("data/ratio_df.json")
replay_columns = ["idAnonymous", "blocks_per_click", "speed", "window_share", "cluster_fit"]

export = {}

for mode in sorted(modes_list):
    mode_rows = ratio_df[ratio_df["fastBuilderMode"] == mode]
    export[mode] = {}

    for cluster in range(mode_rows["cluster"].max() + 1):
        cluster_rows = mode_rows[mode_rows["cluster"] == cluster].sort_values("cluster_fit", ascending=False)
        export[mode][f"cluster {cluster}"] = cluster_rows[replay_columns].round(3).to_dict(orient="records")

    # noise points have no cluster_fit, so we leave that column out and sort them by ID instead
    noise_rows = mode_rows[mode_rows["cluster"] == -1].sort_values("idAnonymous")
    export[mode]["noise"] = noise_rows[replay_columns[:-1]].round(3).to_dict(orient="records")

with open(export_path, "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2)

print("exported", len(ratio_df), "replays to", export_path)